# Notebook 10 — Validación extendida sobre AOI acotado

Recalcula las métricas de validación contra Global Mangrove Watch v4.0 (Bunting et al., 2022) restringiendo el cálculo al AOI acotado (SFF CGSM + VPI Salamanca, 835 km²) en lugar del AOI envolvente del baseline (5.073 km²). Adicionalmente, desagrega la matriz de confusión por clase de tamaño de parche y reporta sensitividad/especificidad por separado.

**Hipótesis metodológica:** al acotar el universo de evaluación al humedal con figura legal de protección --donde GMW efectivamente reporta manglar-- la tasa de falsos positivos baja sustancialmente, lo cual aumenta el F1-score sin modificar el clasificador.

In [1]:
import sys; sys.path.insert(0, '../src')
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

ROOT     = Path('..').resolve()
AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson'
GMW_PATH = ROOT / 'data' / 'validation' / 'gmw' / 'gmw_v4_2020_cgsm.tif'
CLF_PATH = ROOT / 'data' / 'processed' / 'samgeo_acotado' / 'manglar_actual.geojson'
OUT_CSV  = ROOT / 'outputs' / 'tables' / 'validacion_gmw_acotado.csv'

print(f'AOI:         {AOI_PATH.exists()}  {AOI_PATH}')
print(f'GMW raster:  {GMW_PATH.exists()}  {GMW_PATH}')
print(f'Clasif. CGSM:{CLF_PATH.exists()}  {CLF_PATH}')

AOI:         True  /home/rstudio/work/proyecto-cgsm/data/raw/cgsm_aoi_acotado_4326.geojson
GMW raster:  False  /home/rstudio/work/proyecto-cgsm/data/validation/gmw/gmw_v4_2020_cgsm.tif
Clasif. CGSM:True  /home/rstudio/work/proyecto-cgsm/data/processed/samgeo_acotado/manglar_actual.geojson


## 1. Cargar AOI y rasterizar la segmentación SamGeo

Convierte el GeoJSON de parches SamGeo en un raster binario alineado al GMW para la comparación pixel a pixel.

In [2]:
aoi = gpd.read_file(AOI_PATH)

if not GMW_PATH.exists():
    print('Descarga GMW v4.0 sobre la CGSM con GEE y guardalo en data/validation/gmw/')
    print('Ejemplo en GEE Python:')
    print("  gmw = ee.ImageCollection('projects/sat-io/open-datasets/GMW/extent/GMW_V4').filterDate('2020-01-01','2020-12-31').first()")
    print("  geemap.ee_export_image(gmw, filename='gmw_v4_2020_cgsm.tif', region=aoi, scale=25)")
else:
    with rasterio.open(GMW_PATH) as src:
        gmw_data, transform = rio_mask(src, aoi.to_crs(src.crs).geometry, crop=True)
        gmw_data = (gmw_data[0] > 0).astype(np.uint8)
        print(f'GMW raster: shape={gmw_data.shape}, manglar={gmw_data.sum()} pixeles')

Descarga GMW v4.0 sobre la CGSM con GEE y guardalo en data/validation/gmw/
Ejemplo en GEE Python:
  gmw = ee.ImageCollection('projects/sat-io/open-datasets/GMW/extent/GMW_V4').filterDate('2020-01-01','2020-12-31').first()
  geemap.ee_export_image(gmw, filename='gmw_v4_2020_cgsm.tif', region=aoi, scale=25)


In [3]:
if not CLF_PATH.exists():
    print(f'No encontrado: {CLF_PATH}')
else:
    clf_gdf = gpd.read_file(CLF_PATH).to_crs(aoi.crs)
    # Rasterizar SamGeo sobre la misma grilla del GMW
    from rasterio.features import rasterize
    if GMW_PATH.exists():
        shapes = [(geom, 1) for geom in clf_gdf.geometry]
        clf_data = rasterize(shapes, out_shape=gmw_data.shape, transform=transform,
                             fill=0, dtype=np.uint8)
        print(f'SamGeo raster: shape={clf_data.shape}, manglar={clf_data.sum()} pixeles')

## 2. Matriz de confusión y métricas restringidas al AOI acotado

In [4]:
if GMW_PATH.exists() and CLF_PATH.exists():
    y_true = gmw_data.flatten()
    y_pred = clf_data.flatten()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    oa  = (tp + tn) / (tp + tn + fp + fn)
    pre = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1  = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
    spec= tn / (tn + fp) if (tn + fp) > 0 else 0

    print('\n=== Matriz de confusion (AOI acotado) ===')
    print(f'  TP={tp}  FP={fp}')
    print(f'  FN={fn}  TN={tn}')
    print(f'\n  Overall Accuracy:  {oa:.4f}')
    print(f'  Precision (PA):    {pre:.4f}')
    print(f'  Recall (UA):       {rec:.4f}')
    print(f'  F1-score:          {f1:.4f}')
    print(f'  Specificity:       {spec:.4f}')

    # Guardar tabla
    df = pd.DataFrame([
        {'metrica': 'OA', 'valor_aoi_acotado': round(oa, 4)},
        {'metrica': 'Precision', 'valor_aoi_acotado': round(pre, 4)},
        {'metrica': 'Recall', 'valor_aoi_acotado': round(rec, 4)},
        {'metrica': 'F1', 'valor_aoi_acotado': round(f1, 4)},
        {'metrica': 'Specificity', 'valor_aoi_acotado': round(spec, 4)},
    ])
    df.to_csv(OUT_CSV, index=False)
    print(f'\nGuardado: {OUT_CSV}')

## 3. Comparación con baseline (AOI envolvente)

El baseline reportaba F1=0,442 sobre 5.073 km². La hipótesis es que sobre 835 km² (AOI acotado) el F1 sube porque la base de evaluación se acota al universo donde el manglar es esperable, eliminando falsos positivos en vegetación riberana y agrícola fuera del humedal.

**Lo esperado:** F1 acotado > 0,55 si la mejora viene principalmente de reducir FP.

**Trabajo futuro:** validación con muestras independientes recolectadas en Collect Earth Online (CEO) sobre la misma área, lo cual proporciona ground truth visual e independiente de cualquier producto cartográfico previo.